In [1]:
import torch
from transformers import AutoTokenizer, AutoModelForCausalLM

model_id = "Qwen/Qwen2.5-0.5B-Instruct"

tokenizer = AutoTokenizer.from_pretrained(model_id)

model = AutoModelForCausalLM.from_pretrained(
    model_id,
    torch_dtype=torch.float16,
).to("cuda")

model.eval()

print("Model loaded")
print("Parameters:", sum(p.numel() for p in model.parameters()))
print("Device:", next(model.parameters()).device)

c:\Users\simka\anaconda3\envs\llm-lab\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
c:\Users\simka\anaconda3\envs\llm-lab\Lib\site-packages\huggingface_hub\file_download.py:143: UserWarning: `huggingface_hub` cache-system uses symlinks by default to efficiently store duplicated files but your machine does not support them in C:\Users\simka\.cache\huggingface\hub\models--Qwen--Qwen2.5-0.5B-Instruct. Caching files will still work but in a degraded version that might require more space on your disk. This warning can be disabled by setting the `HF_HUB_DISABLE_SYMLINKS_WARNING` environment variable. For more details, see https://huggingface.co/docs/huggingface_hub/how-to-cache#limitations.
To support symlinks on Windows, you either need to activate Developer Mode or to run Python as an administrator. In order to acti

Model loaded
Parameters: 494032768
Device: cuda:0


In [2]:
print(
    "GPU memory allocated:",
    round(torch.cuda.memory_allocated() / 1024**3, 2),
    "GB"
)

GPU memory allocated: 0.93 GB


In [ ]:
text = "The capital of Germany is"

tokens = tokenizer(text, return_tensors="pt") #return as PyTorch tensors

print("Token IDs:", tokens["input_ids"])
print("Tokens:", tokenizer.convert_ids_to_tokens(
    tokens["input_ids"][0]
))

Token IDs: tensor([[ 785, 6722,  315, 9856,  374]])
Tokens: ['The', 'Ġcapital', 'Ġof', 'ĠGermany', 'Ġis']


In [4]:
inputs = tokenizer(
    "The capital of Germany is",
    return_tensors="pt"
).to("cuda")

with torch.no_grad():
    outputs = model(**inputs)

logits = outputs.logits

print("Logits shape:", logits.shape)

Logits shape: torch.Size([1, 5, 151936])


In [5]:
next_token_logits = logits[0, -1, :]

probabilities = torch.softmax(
    next_token_logits.float(), dim=-1
)

top_probs, top_ids = torch.topk(probabilities, k=5)

for token_id, probability in zip(top_ids, top_probs):
    token = tokenizer.decode([token_id.item()])
    print(repr(token), round(probability.item(), 4))

' Berlin' 0.2493
' located' 0.0875
' __' 0.0828
' ______' 0.0766
' the' 0.0458


A correct prediction doesn't necessarily receive a high probability, and high token probability doesn't guarantee factual correctness